# Dose-Response of Cultural-Binding Head Knockout — Instruct Models

Reproduces **Figure 3** and **Table 13** of the paper (instruct models):
the edge-knockout test on the final identified cultural-binding heads
(B→item knockout vs. A→item control, paired t-tests against baseline),
followed by the dose-response experiment where the B→item attention edge
of those heads is scaled by α ∈ {0, 0.25, 0.5, 0.75, 1, 1.5, 2, 3}.
Outputs: the dose-response figure (`dose_response.png`) and a results
pickle (`dose_response_<model>_instruct.pkl`) under
`./results/<model>_instruct/`.

Rebuilt from `pipeline_instruct.ipynb` (canonical master instruct
pipeline), cell 18 ("Stage 4: edge knockout test (final heads) +
dose-response"), with imports and data/model loading taken from cells 2
and 10. The `positions` consumed by the edge-KO half are built exactly
as in the master chain: the Stage 2 span-detection loop of cell 12
(which gates `valid_indices` on span detection succeeding in *both*
conditions, and also extracts the binding attention features as a side
effect; the L1-CV utilities and CV run of that stage are omitted),
followed by the positions builder of cell 15, gated on `valid_indices`..

Run once per `MODEL_KEY`.

In [ ]:
MODEL_KEY = "mistral"  # one of {"mistral", "llama", "gemma2", "nemo"}

In [ ]:
import sys, os
# Locate the repo root (the directory containing common/), whatever the kernel cwd
_p = os.path.abspath(".")
REPO_ROOT = _p if os.path.isdir(os.path.join(_p, "common")) else os.path.abspath("..")
assert os.path.isdir(os.path.join(REPO_ROOT, "common")), (
    "Cannot locate the repo root: run this notebook from its own directory or the repo root")
sys.path.insert(0, REPO_ROOT)

from common import config

CFG = config.init(MODEL_KEY, "instruct")
SEED = config.SEED
DATA_DIR = config.DATA_DIR
HF_TOKEN = config.HF_TOKEN
OUTPUT_DIR = config.OUTPUT_DIR
ACTIVE_MODEL = config.ACTIVE_MODEL  # historical alias used by the source cells

import numpy as np
import torch
import matplotlib.pyplot as plt
from scipy.stats import ttest_rel
from transformers import AutoTokenizer, AutoModelForCausalLM

from common.text_parsers import extract_options
from common.instruct.data import load_n4, build_factorial_as_conditions
from common.instruct.prompts import (format_for_chat, find_option_token_ids,
                                     detect_spans, validate_spans)
from common.instruct.hooks import compute_logit_scores_edge, compute_scores_scaled

In [ ]:
# ── Load data + model (verbatim from pipeline_instruct.ipynb, Stage 1 cell) ──
# ── Load data ──
cultural_items, neutral_items = load_n4(DATA_DIR)
data = build_factorial_as_conditions(cultural_items, seed=SEED)
n_total = len(data['B_cult'])
print(f"  {n_total} examples, {len(set(data['scenarios']))} scenarios")

# ── Load model ──
print(f"\n  Loading {CFG['model_path']}...")
tokenizer = AutoTokenizer.from_pretrained(
    CFG['model_path'], trust_remote_code=True, token=HF_TOKEN)
model = AutoModelForCausalLM.from_pretrained(
    CFG['model_path'], dtype=torch.bfloat16, device_map="auto",
    trust_remote_code=True, token=HF_TOKEN,
    attn_implementation="eager",
)
model.eval()
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

option_tokens_raw = find_option_token_ids(tokenizer)
first_device = next(model.parameters()).device
option_tokens = {opt: torch.tensor(ids, device=first_device)
                 for opt, ids in option_tokens_raw.items()}

for opt, ids in option_tokens_raw.items():
    decoded = [tokenizer.decode([i]) for i in ids]
    print(f"  option '{opt}': {decoded}")

# ── Format texts ──
conditions = ['B_cult', 'B_unrel']
texts_fmt = {c: format_for_chat(data[c], tokenizer) for c in conditions}

# Register runtime singletons so common helpers can see them
config.model = model
config.tokenizer = tokenizer
config.first_device = first_device

In [ ]:
# ── Build valid_indices + positions for the edge-KO test, exactly as in the
# master chain (verbatim from pipeline_instruct.ipynb):
#   * cell 12 — extract_attention_scores + the Stage 2 span-detection loop that
#     builds valid_indices (requires detection in BOTH conditions); the L1-CV
#     utilities and the CV run of that stage are omitted here;
#   * cell 15 — the positions builder gated on valid_indices. ──

# ================================================================
# ATTENTION EXTRACTION
# ================================================================

def extract_attention_scores(model, tokenizer, prompt_text, spans):
    """Extract binding scores for all (layer, head) pairs."""
    enc = tokenizer(prompt_text, return_tensors="pt")
    enc = {k: v.to(first_device) for k, v in enc.items()}
    with torch.no_grad():
        out = model(**enc, output_attentions=True, use_cache=False)
    n_layers = len(out.attentions)
    n_heads_model = out.attentions[0].shape[1]
    seq_len = out.attentions[0].shape[2]
    has_item = spans['item'] is not None
    item_idx = np.array(spans['item']) if has_item else None
    a_idx = np.array(spans['opt_a'])
    b_idx = np.array(spans['opt_b'])
    scores = {
        'bind_a_to_item': np.full((n_layers, n_heads_model), np.nan),
        'bind_b_to_item': np.full((n_layers, n_heads_model), np.nan),
        'bind_avg':       np.full((n_layers, n_heads_model), np.nan),
    }
    for l in range(n_layers):
        attn = out.attentions[l][0].float().cpu().numpy()
        if has_item:
            scores['bind_a_to_item'][l] = attn[:, a_idx][:, :, item_idx].sum(axis=2).mean(axis=1)
            scores['bind_b_to_item'][l] = attn[:, b_idx][:, :, item_idx].sum(axis=2).mean(axis=1)
            scores['bind_avg'][l] = (scores['bind_a_to_item'][l] +
                                     scores['bind_b_to_item'][l]) / 2.0
    del out, enc
    torch.cuda.empty_cache()
    return scores


# ── RUN BINDING CV ──
print("=" * 80)
print(f"STAGE 2: ATTENTION BINDING CV — {CFG['label']}")
print("=" * 80)

all_features = {c: [] for c in conditions}
valid_indices = []
scenarios_valid = []
skipped = 0

for i in range(n_total):
    item_cult = data['items_cult'][i]
    cond_info = {}
    for cond in conditions:
        q_text = data[cond][i].split("\n\n")[0]
        oa, ob = extract_options(q_text)
        cond_info[cond] = {'question': q_text, 'opt_a': oa, 'opt_b': ob, 'item': item_cult}

    prompts, spans_all = {}, {}
    all_ok = True
    for cond in conditions:
        info = cond_info[cond]
        full_text = data[cond][i]
        if CFG["chat_format"] == "gemma":
            prompt = tokenizer.apply_chat_template(
                [{"role": "user", "content": " " + full_text}],
                tokenize=False, add_generation_prompt=True)
        else:
            prompt = tokenizer.apply_chat_template(
                [{"role": "user", "content": full_text}],
                tokenize=False, add_generation_prompt=True)
        enc = tokenizer(prompt, return_tensors="pt")
        ids = enc["input_ids"][0].tolist()
        sp = detect_spans(ids, info['question'], info['opt_a'], info['opt_b'],
                          info['item'], tokenizer, item_required=True)
        if sp is None:
            all_ok = False; break
        prompts[cond] = prompt
        spans_all[cond] = sp
        del enc
    if not all_ok:
        skipped += 1; continue
    if len(valid_indices) < 3:
        for cond in conditions:
            enc = tokenizer(prompts[cond], return_tensors="pt")
            validate_spans(enc["input_ids"][0], spans_all[cond], tokenizer,
                         label=f"{cond}: {cond_info[cond]['opt_a']} vs {cond_info[cond]['opt_b']}")
            del enc
    for cond in conditions:
        scores = extract_attention_scores(model, tokenizer, prompts[cond], spans_all[cond])
        all_features[cond].append(scores)
    valid_indices.append(i)
    scenarios_valid.append(data['scenarios'][i])
    if len(valid_indices) % 50 == 0:
        print(f"  Processed {len(valid_indices)}/{n_total} (skipped {skipped})")

print(f"\n  Total valid: {len(valid_indices)} / {n_total} (skipped {skipped})")


# ── Build positions for edge knockout ──
texts_fmt = {cond: format_for_chat(data[cond], tokenizer) for cond in conditions}
positions = {c: [] for c in conditions}

for c in conditions:
    for i in range(n_total):
        if i not in valid_indices:
            positions[c].append(None); continue
        q_text = data[c][i].split("\n\n")[0]
        oa, ob = extract_options(q_text)
        item = data['items_cult'][i]
        enc = tokenizer(texts_fmt[c][i], return_tensors="pt")
        ids = enc["input_ids"][0].tolist()
        sp = detect_spans(ids, q_text, oa, ob, item, tokenizer, item_required=True)
        if sp is None:
            positions[c].append(None)
        else:
            assoc_pos = data['assoc_pos'][i]
            B_tokens = sp['opt_a'] if assoc_pos == 'a' else sp['opt_b']
            A_tokens = sp['opt_b'] if assoc_pos == 'a' else sp['opt_a']
            positions[c].append({
                'item_tokens': sp['item'], 'B_tokens': B_tokens,
                'A_tokens': A_tokens, 'B_text': oa if assoc_pos == 'a' else ob,
            })
        del enc

In [ ]:
# ================================================================
# EDGE KO TEST — final heads from CFG (or override from CV)
# ================================================================

FINAL_HEADS = CFG['heads']  # override here if CV discovered different heads

print("=" * 80)
print(f"STAGE 4: EDGE KO TEST + DOSE-RESPONSE — {CFG['label']}")
print(f"  Heads: {FINAL_HEADS}")
print("=" * 80)

# ── Baseline ──
print("\n  Computing baseline...")
scores_base = {}
for cond in conditions:
    scores_base[cond] = compute_logit_scores_edge(
        model, tokenizer, texts_fmt[cond], data[cond],
        {}, positions[cond], 'B_to_item', option_tokens)
delta_base = scores_base['B_cult'].mean() - scores_base['B_unrel'].mean()
diffs_base = scores_base['B_cult'] - scores_base['B_unrel']
print(f"  Baseline Δ(S) = {delta_base:.4f}")

# ── B→item KO ──
print("\n  B→item KO...")
scores_ko = {}
for cond in conditions:
    scores_ko[cond] = compute_logit_scores_edge(
        model, tokenizer, texts_fmt[cond], data[cond],
        FINAL_HEADS, positions[cond], 'B_to_item', option_tokens)
delta_ko = scores_ko['B_cult'].mean() - scores_ko['B_unrel'].mean()
diffs_ko = scores_ko['B_cult'] - scores_ko['B_unrel']
t, p = ttest_rel(diffs_base, diffs_ko)
print(f"  B→item KO Δ(S) = {delta_ko:.4f}  (reduction: {(1-delta_ko/delta_base)*100:.1f}%)")
print(f"  t = {t:.3f}, p = {p:.6f}")

# ── A→item KO (control) ──
print("\n  A→item KO (control)...")
scores_ctrl = {}
for cond in conditions:
    scores_ctrl[cond] = compute_logit_scores_edge(
        model, tokenizer, texts_fmt[cond], data[cond],
        FINAL_HEADS, positions[cond], 'A_to_item', option_tokens)
delta_ctrl = scores_ctrl['B_cult'].mean() - scores_ctrl['B_unrel'].mean()
diffs_ctrl = scores_ctrl['B_cult'] - scores_ctrl['B_unrel']
t_c, p_c = ttest_rel(diffs_base, diffs_ctrl)
print(f"  A→item KO Δ(S) = {delta_ctrl:.4f}  (reduction: {(1-delta_ctrl/delta_base)*100:.1f}%)")
print(f"  t = {t_c:.3f}, p = {p_c:.6f}  (should be NS)")


# ================================================================
# DOSE-RESPONSE
# ================================================================
print(f"\n{'='*80}")
print("DOSE-RESPONSE")
print(f"{'='*80}")

# ── Build positions for dose-response ──
texts_fmt_dr = {c: format_for_chat(data[c], tokenizer) for c in conditions}
positions_dr = {c: [] for c in conditions}
for c in conditions:
    for i in range(n_total):
        q_text = data[c][i].split("\n\n")[0]
        oa, ob = extract_options(q_text)
        item = data['items_cult'][i]
        enc = tokenizer(texts_fmt_dr[c][i], return_tensors="pt")
        ids = enc["input_ids"][0].tolist()
        sp = detect_spans(ids, q_text, oa, ob, item, tokenizer, item_required=True)
        if sp is None:
            positions_dr[c].append(None)
        else:
            assoc_pos = data['assoc_pos'][i]
            B_tokens = sp['opt_a'] if assoc_pos == 'a' else sp['opt_b']
            A_tokens = sp['opt_b'] if assoc_pos == 'a' else sp['opt_a']
            positions_dr[c].append({
                'item_tokens': sp['item'], 'B_tokens': B_tokens,
                'A_tokens': A_tokens,
            })
        del enc

ALPHAS = [0.0, 0.25, 0.5, 0.75, 1.0, 1.5, 2.0, 3.0]
results_dr = {}
for alpha in ALPHAS:
    print(f"\n  α = {alpha:.2f} ...")
    scores = {}
    for c in conditions:
        scores[c] = compute_scores_scaled(
            model, tokenizer, texts_fmt_dr[c], positions_dr[c],
            FINAL_HEADS, 'B_to_item', option_tokens, alpha=alpha)
    delta = scores['B_cult'].mean() - scores['B_unrel'].mean()
    results_dr[alpha] = {'delta': delta, 'scores': scores}
    red = (1 - delta / delta_base) * 100 if abs(delta_base) > 1e-10 else 0
    print(f"    Δ(S) = {delta:.4f}  (reduction: {red:.1f}%)")

# ── Plot ──
fig, ax = plt.subplots(1, 1, figsize=(8, 5))
alphas_list = sorted(results_dr.keys())
deltas_list = [results_dr[a]['delta'] for a in alphas_list]
ax.plot(alphas_list, deltas_list, 'o-', linewidth=2, markersize=8)
ax.axhline(y=delta_base, color='gray', linestyle=':', label=f'Baseline Δ(S)={delta_base:.3f}')
ax.axvline(x=1.0, color='gray', linestyle=':', alpha=0.3)
ax.set_xlabel('α', fontsize=12)
ax.set_ylabel('Δ(S)', fontsize=12)
ax.set_title(f'{CFG["label"]}: Dose-Response', fontsize=13)
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "dose_response.png", dpi=150)
plt.show()

In [ ]:
# ── Persistence (ADDED for the submission repo; strictly additive) ──
# The figure itself is already saved by the verbatim cell above
# (OUTPUT_DIR / "dose_response.png").
import pickle

dose_response_results = {
    'model': ACTIVE_MODEL,
    'label': CFG['label'],
    'final_heads': FINAL_HEADS,
    'alphas': ALPHAS,
    'delta_base': delta_base,
    'diffs_base': diffs_base,
    'delta_ko': delta_ko,
    'diffs_ko': diffs_ko,
    't_ko': t, 'p_ko': p,
    'delta_ctrl': delta_ctrl,
    'diffs_ctrl': diffs_ctrl,
    't_ctrl': t_c, 'p_ctrl': p_c,
    'results_dr': results_dr,
}
pkl_path = OUTPUT_DIR / f"dose_response_{ACTIVE_MODEL}_instruct.pkl"
with open(pkl_path, 'wb') as f:
    pickle.dump(dose_response_results, f)
print(f"Saved dose-response results to {pkl_path}")